# Router playground — design arm

One place to train the design-arm router (bge embeddings + char-ngram SVD, cell and
corpus branches) with every knob visible, watch the train/validation curves, and fire
your own smoke queries at the result.

The split, the SVD fit, the z-scoring, and the threshold tuning all respect the
fit/val boundary — unlike the old section-8 smoke test, nothing here is tuned
in-sample. Labels are still shakedown tier (see `encoder_experiments.ipynb` intro).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# evaluate first: it loads lightgbm before model's torch (macOS OpenMP guard)
from encoder_router.evaluate import BGE, tuned_thresholds
from encoder_router.model import EncoderRouter
from encoder_router.table import NgramSvd
from encoder_router.training import QueryEmbeddings, TrainingTable
from encoder_router.table import (
    HEAD_ROUTES,
    HEDGE,
    ROUTES,
    serve_from_probabilities,
)

pd.set_option("display.width", 160)
table = TrainingTable()
frame = table.frame
print(f"{len(frame):,} rows across {frame['dataset'].nunique()} lanes")
print(f"heads: {HEAD_ROUTES}, hedge: {HEDGE}")

## 1 · Hyperparameters

Everything tunable lives in this one cell. `PARAMS` goes verbatim into
`EncoderRouter(**PARAMS)` — see the dataclass in `encoder_router/model.py` for
what each field does.

In [ ]:
SPLIT_SEED = 0       # pins the val split and the SVD — keep fixed so runs stay comparable
SEED = None          # training randomness (init + batch order); None = fresh every run
SEED = SEED if SEED is not None else int(np.random.default_rng().integers(2**31))
print(f"run seed: {SEED}  (put this in SEED to replay the run exactly)")

VAL_FRACTION = 0.2  # held out for early stopping, threshold tuning, and readout
TIE_WEIGHT = 1    # all_tied rows in the route loss (the v3 sweep setting; 1.0 = full)

PARAMS = dict(
    latent=128, hidden=256, dropout=0.2,   # capacity
    lr=1e-4, weight_decay=1e-5, batch_size=1048,
    epochs=500, patience=40,               # early stopping on val route loss
    lambda_cell=0.5, lambda_corpus=0.5,    # branch loss weights
    anneal_epochs=20,                      # branch losses fade to zero over this many epochs
    seed=SEED,
)

## 2 · The split

A random query-level holdout. Every lane appears on both sides, so this measures
in-distribution fit — for generalization to unseen corpora, the leave-one-lane-out
sweep in `encoder_experiments.ipynb` is the instrument.

In [ ]:
rng = np.random.default_rng(SPLIT_SEED)
order = rng.permutation(len(frame))
cut = max(int(len(frame) * VAL_FRACTION), 1)
val_idx, fit_idx = np.sort(order[:cut]), np.sort(order[cut:])
train_mask = np.zeros(len(frame), dtype=bool)
train_mask[fit_idx] = True

val_frame = frame.iloc[val_idx].reset_index(drop=True)
pd.DataFrame({
    "rows": [len(fit_idx), len(val_idx)],
    "lanes": [
        frame.iloc[fit_idx]["dataset"].nunique(),
        val_frame["dataset"].nunique(),
    ],
    "routes_differ": [
        float((frame.iloc[fit_idx]["shape"] == "routes_differ").mean()),
        float((val_frame["shape"] == "routes_differ").mean()),
    ],
}, index=["fit", "val"]).round(3)

## 3 · Inputs and targets (design arm)

Inputs: cached bge embeddings + char 3-5-gram SVD **fit on the fit rows only**.
Targets: two route heads, cell multi-hot for the cell branch, and the corpus block
(profiles + fit-side outcome rates) z-scored with fit-side stats.

In [ ]:
embeddings = QueryEmbeddings(BGE).matrix(frame)
svd = NgramSvd(seed=SPLIT_SEED).fit(frame.loc[train_mask, "query"])
x = np.concatenate(
    [embeddings, svd.transform(frame["query"])], axis=1
).astype(np.float32)

route = table.route_targets().to_numpy(dtype=np.float32)
cell = table.cell_targets.to_numpy(dtype=np.float32)
joined = pd.concat(
    [table.corpus_targets(), table.outcome_rates(train_mask)], axis=1
)
stats = joined[train_mask]
corpus = (
    (joined - stats.mean()) / stats.std().replace(0.0, 1.0).fillna(1.0)
).to_numpy(np.float32)
weights = np.where(
    frame["shape"] == "all_tied", TIE_WEIGHT, 1.0
).astype(np.float32)
x.shape, route.shape, cell.shape, corpus.shape

## 4 · Train

The tqdm postfix streams train/val route loss live; `router.history` keeps the
full trace. Early stopping restores the best-val-loss weights.

In [ ]:
router = EncoderRouter(**PARAMS).fit(
    x[fit_idx], route[fit_idx], cell[fit_idx], corpus[fit_idx],
    route_weights=weights[fit_idx],
    x_val=x[val_idx], val_route_targets=route[val_idx],
    val_route_weights=weights[val_idx],
)
history = pd.DataFrame(router.history)
best = history.loc[history["val"].idxmin()]
print(
    f"stopped after epoch {int(history['epoch'].iloc[-1])}; "
    f"best val loss {best['val']:.4f} at epoch {int(best['epoch'])} "
    f"(weights restored there)"
)

In [ ]:
ROUTER_PATH = './data/router-encoder'
router.save(ROUTER_PATH)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history["epoch"], history["train"], label="train")
ax.plot(history["epoch"], history["val"], label="validation")
ax.axvline(
    best["epoch"], color="grey", ls="--", lw=1,
    label=f"restored epoch {int(best['epoch'])}",
)
ax.set_xlabel("epoch")
ax.set_ylabel("masked route BCE")
ax.legend()
ax.set_title("route-head loss")
plt.tight_layout()
plt.show()

## 5 · Head calibration on validation

With `pos_weight = neg/pos`, an uninformative head settles at exactly 0.5 — spread
away from 0.5 is learned signal, a spike at 0.5 is a head that learned nothing.

In [ ]:
val_probs = router.probabilities(x[val_idx])
fig, axes = plt.subplots(
    1, len(HEAD_ROUTES), figsize=(10, 3), sharey=True
)
for ax, head in zip(np.atleast_1d(axes), HEAD_ROUTES):
    ax.hist(val_probs[head], bins=40)
    ax.axvline(0.5, color="grey", ls="--", lw=1)
    ax.set_title(f"p({head} acceptable)")
plt.tight_layout()
plt.show()
val_probs.describe().round(3)

## 6 · Thresholds and validation readout

Thresholds are tuned on the **validation** probabilities (the fit side never sees
them) to maximize raw captured score — no cost discount. Serve rule: the most
probable head among those clearing their thresholds; neither fires → the rrf hedge.

In [ ]:
thresholds = tuned_thresholds(val_probs, val_frame)
print("tuned thresholds:", dict(zip(HEAD_ROUTES, thresholds.round(2))))

judged = (
    val_frame
    .assign(served=serve_from_probabilities(val_probs, thresholds))
    .dropna(subset=["serve"])
)
differ = judged[judged["shape"] == "routes_differ"]
scores = {r: judged[f"score_{r}"].to_numpy() for r in ROUTES}
pick = lambda routes: np.array(  # noqa: E731
    [scores[r][i] for i, r in enumerate(routes)]
)
pd.Series({
    "serve_agreement": (judged["served"] == judged["serve"]).mean(),
    "differ_agreement": (differ["served"] == differ["serve"]).mean(),
    "captured": pick(judged["served"]).mean(),
    "serve_captured": pick(judged["serve"]).mean(),
    "oracle": np.column_stack(list(scores.values())).max(axis=1).mean(),
    **{f"served_{r}": (judged["served"] == r).mean() for r in ROUTES},
}).astype(float).round(4)

## 7 · Smoke queries

Edit `SMOKE_QUERIES` and re-run the last cell — the seven standing archetype probes
are the default. `expected`/`agrees` fill in only for queries that are probes.
When both heads clear their thresholds, the more probable one is served — cost
never breaks the tie.

In [ ]:
from sentence_transformers import SentenceTransformer

from encoder_router.table import EMBEDDING_PREFIXES
from hybrid_search_rrf_dataset.probes import ARCHETYPE_PROBES

encoder = SentenceTransformer(BGE)


def route_queries(queries: list[str]) -> pd.DataFrame:
    emb = encoder.encode(
        [EMBEDDING_PREFIXES[BGE] + q for q in queries],
        normalize_embeddings=True,
    )
    qx = np.concatenate(
        [emb, svd.transform(pd.Series(queries))], axis=1
    ).astype(np.float32)
    probs = router.probabilities(qx)
    out = probs.round(3)
    out.insert(0, "query", queries)
    out["served"] = serve_from_probabilities(probs, thresholds)
    return out

In [ ]:
SMOKE_QUERIES = [p.query for p in ARCHETYPE_PROBES] + [
    "excitement out of enjoying math",
    "recommend epic poetry similar to the iliad and the odyssey",
    "recommend me books with a similar feeling to what i love"
]

smoke = route_queries(SMOKE_QUERIES)
expected = {p.query: p.expected for p in ARCHETYPE_PROBES}
smoke["expected"] = [
    getattr(expected.get(q), "value", None) for q in smoke["query"]
]
smoke["agrees"] = [
    None if e is None else e == s
    for e, s in zip(smoke["expected"], smoke["served"])
]
smoke